<figure>
  <IMG SRC="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Fachhochschule_Südwestfalen_20xx_logo.svg/320px-Fachhochschule_Südwestfalen_20xx_logo.svg.png" WIDTH=250 ALIGN="right">
</figure>

# Machine Learning
### Sommersemester 2026
Prof. Dr. Stefan Goetze

# Neuronale Netze 

In diesem Aufgabenblatt wollen wir nochmal auf die Logistische Regression zurückblicken und aus dem Linearen Modell ein eifaches neuronales Netz herleiten.

## Rückblick Logistische Regression

Dazu schauen wir uns nochmal im Detail an, wie die Modell- und Kostenfunktionen der logistischen Regression aufgebaut sind und implementieren das Gradientenverfahren zur Minimierung der Kostenfunktion per Hand (d.h. mit Python bzw. NumPy).

Bei der **Logistischen Regression** haben wir das Gradientenverfahren benutzt, um die Parameter unseres Modells, einer linearen Funktion 

$\mathbf{z} = f_{\boldsymbol{\theta}}(\mathbf{x})=\theta_0 + \boldsymbol{\theta}_1^T \mathbf{x}$

transformiert durch die Aktivierungsfunktion 

$\hat{\mathbf{y}} = h_{\boldsymbol{\theta}}(\mathbf{x}) = \sigma(\mathbf{z}) = \frac{1}{1+e^{-\mathbf{z}}}$

schrittweise zu verbessern.

Dabei war $\mathbf{x}=\left[x_1, x_2, ... , x_n\right]^T$ ein Datenpunk oder *Merkmalsvektor* mit $n$ Merkmalen (*Features*), und $\boldsymbol{\theta}=\left[\theta_0, \boldsymbol{\theta}_1^T\right]^T$ waren die Parameter des Regressionsmodells, aufgeteilt in den Bias $\theta_0$ und die übrigen Modellparameter $\boldsymbol{\theta}_1=\left[\theta_1, \theta_2, ... , \theta_n\right]^T$.

Die Verbesserung, bzw. die Qualität des Modells, haben wir anhand der Kostenfunktion 

$J_{\boldsymbol{\theta}}(\mathbf{x})=-\frac{1}{m} \sum\limits_{i = 1}^{m} \left[y^{(i)}\log\left(\hat y^{(i)}\right) + \left(1-y^{(i)}\right)\log\left(1- \hat y^{(i)}\right)\right]$

berechnet.

Im folgenden werden wir die logistische Regression von hand implementieren, um diese dann mit einer Implementierung im [Keras Framework](https://keras.io/) zu vergleichen. 

Wir starten, wie üblich mit dem Import der benötigten Bibliotheken.

In [1]:
import numpy as np                                    # math
import matplotlib.pyplot as plt                       # visualisation
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler

Als nächstes Initialisieren wir (zunächst aus Zufallszahlen)

- Einen (Trainings-) Datensatz als Matrix $\mathbf{X} \in \mathbb{R}^{m\times{}n}$, also mit $m$ Datenpunkten und Zählindex $i$ ($1 \leq i \leq m$) als Zeilenvektoren $\left(\mathbf{x}^{(i)}\right)^T = \left[ x_1^{(i)}, x_2^{(i)}, ..., x_n^{(i)} \right]$, die jeweils eine $n$ Merkmale beinhalten, also eine Länge von $n$ haben. Die Matrix des Trainingsdatensatzes sieht damit wie folgt aus:

$\begin{equation}
  \mathbf{X} =
  \left[
  \begin{array}{c}
  \left(\mathbf{x}^{(1)}\right)^T \\
  \left(\mathbf{x}^{(2)}\right)^T \\
  \vdots    \\
  \left(\mathbf{x}^{(m)}\right)^T \\
  \end{array} \\
  \right] \\
  = \\
  \left[ \\
  \begin{array}{cccc} \\
  x_1^{(1)} & x_2^{(1)} & ...    & x_n^{(1)} \\
  x_1^{(2)} & x_2^{(2)} & ...    & x_n^{(2)} \\
  \vdots    & \vdots    & \ddots & \vdots    \\
  x_1^{(m)} & x_2^{(m)} & ...    & x_n^{(m)} \\
  \end{array} \\
  \right] \in \mathbb{R}^{m\times{}n} \\
\end{equation}$
- Die Parameter des Modells $\boldsymbol{\theta}=\left[\theta_0, \boldsymbol{\theta}_1^T\right]^T$. Dabei ist der Bias Parameter $\theta_0$ ein skalarer Wert, $\boldsymbol{\theta}_1$ hingegen ein Vektor aus $\mathbb{R}^n$.

In [2]:
# Test Case
m_c, n_c = 20, 4                                  # input feature dimensions, m: feature vector length, n: no of feat. vectors / data points
np.random.seed(0)                                 # fix seed for random genertor
X_c = np.random.randn(m_c, n_c)                   # train data
theta0_c, theta1_c = 0, np.random.randn(n_c, 1)   # parameters
y_c = np.random.randn(m_c, 1)                     # labels

In [ ]:
# (optional:) get familiar with the data

#print('Randomly initialised training data matrix X of shape ' + str(X_c.shape) + 
#      ' (m='+str(m_c)+' datapoints (feature vectors) of length n='+str(n_c)+' each):')
#print(X_c)
#print('Parameter vector theta0:')
#print(theta0_c)
#print('Parameter vector theta1:')
#print(theta1_c)
#print('Parameter vector theta: ')
#print(np.vstack((theta0_c, theta1_c)))
#X1=X_c[0,:]
#print(X1)

#print(X1@theta1_c)

### Modellfunktion

<div class="alert alert-block alert-info">
    
**Aufgabe: Schreibe eine Funktion $f(\cdot)$, die folgende Parameter erhält:**

Die Matrix $\mathbf{X} \in \mathbb{R}^{m\times{}n}$, die die $m$ Datenpunkte des Trainigsdatensatzes enthält.

Die Parameter $\boldsymbol{\theta}$. Dabei ist der Bias Parameter $\theta_0$ ein skalarer Wert, $\boldsymbol{\theta}_1$ hingegen ein Vektor aus $\mathbb{R}^n$, in Numpy also mit der Dimension <code>(n, 1)</code>.


**$f(\cdot)$ soll folgende lineare Funktion implementieren:** 
$$\mathbf{z} = f_{\boldsymbol{\theta}}(\mathbf{X})=\theta_0 + \mathbf{X} \boldsymbol{\theta}_1$$

*Hinweis: Denken Sie daran, dass Matrix-Multiplikationen in NumPy mit dem @-Operator implementiert werden können.*
</div>

In [7]:
def f(X, theta0, theta1):
    """evaluates linear function.
    Arguments:
        X: value/data
        theta0: Bias
        theta1: Function slope
    """
    
    z = X@theta1 + theta0

    return z

In [20]:
# Test Cell
#----------

z = f(X_c, theta0_c, theta1_c)
#----------
# f

assert z.shape == (m_c, 1), 'Use correctly sequenced matrix multiplication'
assert np.isclose(z[0], -4.68171, atol=0.001), 'Expected -4.68171 but got %.5f' %z[0]

del z

<div class="alert alert-block alert-info">
    
**Aufgabe: Implementieren Sie die Perzepton-Fuinkion $h(\cdot)$. Die Funktion $h(\cdot)$ soll die gleichen Parameter wie $f(\cdot)$ erhalten und die Funktion $f(\cdot)$ intern aufrufen. Das Ergebnis von $f(\cdot)$ soll durch die Sigmoid-Aktivierungsfunktion transformiert werden.:** 
$$\hat{\mathbf{y}} = h_{\boldsymbol{\theta}}(\mathbf{X}) = \sigma(\mathbf{z}) = \frac{1}{1+e^{-\mathbf{z}}}$$

*Hinweis: Die Exponentialfunktion angewendet auf ein Argument $z$ können Sie in NumPy mit dem Methodenaufruf `np.exp(z)` berechnen.*
</div>

In [18]:
def h(X, theta0, theta1):
    """returns the sigmoid of the linear function.
    Arguments:
        X: Data
        theta0: Bias
        theta1: Function slope
    """
    
    z = f(X, theta0=theta0, theta1=theta1)

    y_hat = 1/(1+np.exp(-z))
    
    return y_hat

In [21]:
# Test Cell
#----------

y_hat = h(X_c, theta0_c, theta1_c)
#----------
# h

assert y_hat.shape == (m_c, 1)
assert np.isclose(y_hat[0], 0.00918, atol=0.001), 'Expected 0.00918 but got %.5f' %y_hat[0]

del y_hat

## Kostenfunktion

<div class="alert alert-block alert-info">
    
**Aufgabe: Berechnen Sie nun die Kostenfunktion. Schreiben Sie eine Funktion `J()`, die folgende Parameter erhält:**

- Die Matrix $\mathbf{X} \in \mathbb{R}^{m\times{}n}$, die die Datenpunkte des Trainigsdatensatzes enthält.
- Die Parameter $\boldsymbol{\theta}$. Dabei ist der Bias Parameter $\theta_0$ ein skalarer Wert, $\boldsymbol{\theta}_1$ hingegen ein Vektor aus $\mathbb{R}^n$, in Numpy also mit der Dimension <code>(n, 1)</code>.
- Die Label $\mathbf{y}$ in der Größe des Datensatzes <code>(m, 1)</code>.


<b>$J(\cdot )$ soll die folgende Kostenfunktion berechnen ([Binäre Kreuzentropie](https://towardsdatascience.com/understanding-binary-cross-entropy-log-loss-a-visual-explanation-a3ac6025181a/)):</b> 
    
$$J_{\boldsymbol{\theta}}(x)=-\frac{1}{m} \sum\limits_{i = 1}^{m} [y^{(i)}\log(\hat y^{(i)}) + (1-y^{(i)})\log(1- \hat y^{(i)})]$$
</div>

*Hinweis: Sie können die Summe in zwei Teilsummen aufteilen und so die Berechnung vektorisieren:*

 $$-\frac{1}{m} \sum\limits_{i = 1}^{m} [y^{(i)}\log(\hat y^{(i)}) + (1-y^{(i)})\log(1- \hat y^{(i)})] = -\frac{1}{m}  \left( \sum\limits_{i = 1}^{m} y^{(i)}\log(\hat y^{(i)}) + \sum\limits_{i = 1}^{m}(1-y^{(i)})\log(1- \hat y^{(i)})\right)$$
 
*und $\sum\limits_{i = 1}^{m} y^{(i)}\log(\hat y^{(i)})$ ist einfach das Skalarprodukt aus $\mathbf{y}$ und $\log(\hat{\mathbf{y}})$.*

*Den natürlichen Logarithmus von `x` berechnen Sie in NumPy mit `np.log(x)`, das Skalarprodukt mit `np.dot(a,b)` oder, falls `a` und `b` (2-dimensionale) Spaltenvektoren sind, einfach mit der Matrizenmultiplikation `a.T@b` (`a` wird transponiert, um einen Zeilenvektor zu erhalten).*


In [ ]:
def J(X: np.ndarray, theta0, theta1, y):
    """computes the Cross-entropy cost function
    Arguments:
        X: Data
        theta0: Bias
        theta1: Function slope
        y: True labels
    """
    m = X.shape[0]

    y_hat = h(X, theta0, theta1)
    
    J_1 = np.dot(y,np.log(y_hat))

    J_2 = np.dot((1-y),(1-y_hat.T))

    J = -1/m*(J_1 + J_2)
    
    return J.squeeze()

In [29]:
# Test Cell
#----------

cost = J(X_c,theta0_c,theta1_c,y_c)
print(cost)
#----------
# J

assert cost.shape == (), 'Use correctly sequenced matrix multiplication'
assert np.isclose(cost, 1.49867, atol=0.001), 'Expected 1.49867 but got %.5f' %cost

del cost

[[ 3.73252900e-01  2.06131295e-01  1.58103450e-01  1.20527667e-01
   8.60093195e-02  5.96599769e-04  2.97007346e-01  1.13863249e-01
   1.24716748e-01  6.66969088e-02  3.11877249e-01  1.51660516e-01
   3.87866680e-02  1.23947549e-01  4.59714365e-02  6.17850756e-03
   1.82414992e-01  2.42814502e-02  1.27681946e-01  5.30913973e-02]
 [ 4.89057512e-01  2.75127828e-01  2.12946567e-01  1.63771775e-01
   1.17988043e-01  8.42499072e-04  3.91705419e-01  1.54984318e-01
   1.69284058e-01  9.20398628e-02  4.10710902e-01  2.04554186e-01
   5.40234723e-02  1.68272524e-01  6.38732247e-02  8.70699435e-03
   2.44496566e-01  3.39933027e-02  1.73180832e-01  7.35896009e-02]
 [ 2.85335566e-01  1.53750036e-01  1.16467279e-01  8.76973105e-02
   6.17314944e-02  4.09916260e-04  2.25113817e-01  8.26446759e-02
   9.08818344e-02  4.74568722e-02  2.36844043e-01  1.11504340e-01
   2.72190870e-02  9.02966117e-02  3.23806350e-02  4.25891373e-03
   1.35283479e-01  1.69083398e-02  9.31397908e-02  3.75294314e-02]
 [-1.00

AssertionError: Use correctly sequenced matrix multiplication

## Gradientenverfahren

<div class="alert alert-block alert-info">
    
**Aufgabe: Schreiben Sie für das Gradientenverfahren eine Funktion `grads()`, die folgende Parameter erhält.**
<ul>
<li>Die Matrix $\mathbf{X} \in \mathbb{R}^{m\times{}n}$, die die Datenpunkte des Trainigsdatensatzes enthält.
</li>
<li>Die Parameter $\mathbf{\theta}$. Dabei ist der Bias Parameter $\theta_0$ ein skalarer Wert, $\mathbf{\theta}_1$ hingegen ein Vektor aus $\mathbb{R}^n$, in Numpy also mit der Dimension <code>(n, 1)</code>.
</li>
<li>Die Label $\mathbf{y}$ in der Größe des Datensatzes <code>(m, 1)</code>.
</li>
    
</ul>

**und den Gradienten $\partial \boldsymbol{\theta}$ für die Parameter $\boldsymbol{\theta}$ berechnet** 
</div>

Dabei ist $\partial\theta_0$ ein Skalar der dem Gradienten des Bias Parameters entspricht: 
$$\partial\theta_0 = \frac{1}{m} \sum_{i=1}^m (\hat y^{(i)}-y^{(i)})$$

$\partial\boldsymbol{\theta}_1$ ist ein Vektor der Dimension `(n, 1)` mit den Gradienten der anderen Parameter: 
$$ \partial \boldsymbol{\theta}_1 = \frac{1}{m}\mathbf{X}^T(\hat{\mathbf{y}}-\mathbf{y})$$

In [ ]:
def grads(X,theta0, theta1,y):
    """Calculates grads of the cost function with respect to the parameters.
    Arguments:
        X: Data
        theta0: Bias
        theta1: Function slope
        y: True labels
    """
    
    # YOUR CODE HERE
    raise NotImplementedError()
    
    return dtheta0, dtheta1

In [ ]:
# Test Cell
#----------

dt0, dt1 = grads(X_c,theta0_c,theta1_c,y_c)
#----------
# grads

assert dt0.shape == (), 'theta0 should be a scaler'
assert dt1.shape == theta1_c.shape
assert np.isclose(dt1[0], -0.22445, atol=0.001), 'Expected -0.22445 but got %.5f' %dt1[0]

<div class="alert alert-block alert-info">
    
**Aufgabe: Schreiben Sie nun eine Funktion, die die Modellparameter aufgrund der berechneten Gradienten aktualisiert. Die Funktion `update()`erhält die Parameter $\boldsymbol{\theta}$, die Gradienten $\partial \boldsymbol{\theta}$ sowie die Lernrate $\alpha$ und berechnet:**

$$\boldsymbol{\theta}_{\mathrm{new}} = \boldsymbol{\theta}_{\mathrm{old}} - \alpha \cdot \partial \boldsymbol{\theta}$$
</div>

In [ ]:
def update(theta0, theta1,dtheta0, dtheta1, alpha):
    """updates parameters using gradient decent updating rule."""
    
    # YOUR CODE HERE
    raise NotImplementedError()
    
    return theta0, theta1

In [ ]:
# Test Cell
#----------

t0, t1 = update(theta0_c, theta1_c,dt0, dt1, 0.1)
#----------
# update

assert t0.shape == (), 'theta0 should be a scaler'
assert t1.shape == theta1_c.shape
assert np.isclose(t1[0], -1.14270, atol=0.001), 'Expected -0.141491 but got %.5f' %t1[0]
del t0, t1, dt0, dt1

Nun können wir das iterative Gradientenverfahren programmieren.

<div class="alert alert-block alert-info">
    
**Aufgabe: Schreiben Sie eine Funktion `gradient_descent()`, die folgende Parameter erhät:**
1. Die Matrix $\mathbf{X} \in \mathbb{R}^{m\times{}n}$, die die Datenpunkte des Trainigsdatensatzes enthält. .
2. Die Parameter $\boldsymbol{\theta}$. Dabei ist der Bias Parameter $\theta_0$ ein skalarer Wert, $\boldsymbol{\theta}_1$ hingegen ein Vektor aus $\mathbb{R}^n$, in Numpy also mit der Dimension `(n, 1)`.
3. Die Label $\mathbf{y}$ in der Größe des Datensatzes `(m, 1)`.
4. Die Lernrate $\alpha$.
5. Die Anzahl der Iterationen.

**Die Funktion soll die Trainierten Modellparameter $\boldsymbol{\theta}$ zurückgeben.**
</div>

*Hinweis*: Berechnen Sie Kosten mit der Funktion `J()` und hängen Sie diese Kosten nach jedem Berechnungsschritt and die Liste `cost` an --> Berechnen Sie die Gradienten mit der Funktion `grads()` --> Verwenden Sie diese Gradienten um die Parameter mit der Funktion `optim()`

In [ ]:
def gradient_decent(X, theta0, theta1, y, alpha=0.1, iterations=400):
    """performs gradient decent optimization.
    Arguments:
        X: Data
        theta0: Bias
        theta1: Function slope
        y: True labels
        alpha(default=0.1): Learning rate
        iterations(default=400): number of updating iterations
    """
    
    costs = []
    
    for i in range(iterations):
        # YOUR CODE HERE
        raise NotImplementedError()
        
    return theta0, theta1, costs

In [ ]:
# Test Cell
#----------

t0, t1, costs = gradient_decent(X_c, theta0_c, theta1_c, y_c)
#----------
# gradient_decent

assert len(costs) == 400, 'Make sure to calculate and append the cost in every iteration. Expected length of vector cost is 400.'
assert np.isclose(t1[3], 2.93724, atol=0.001), 'Expected 1.05186 but got %.5f' %t1[3]

del t0, t1, costs

## Beispiel

Wir habe nun alle Funktionen um unser Perzeptron für einen *echten* Datensatz zu einzusetzen.
Wir verwenden hier den Brustkrebs-Datensatz aus Sklearn:

In [ ]:
scaler = MinMaxScaler()
data = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(data.data,data.target,test_size=0.3)

# preprocessing
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
y_train = np.expand_dims(y_train, 1)
y_test = np.expand_dims(y_test, 1)

In [ ]:
# initializing parameters
theta0 = 0.
theta1 = np.zeros((len(X_train[0]), 1))

#training the model
theta0, theta1, costs = gradient_decent(X_train, theta0, theta1, y_train.reshape(-1, 1), alpha = 2)

In [ ]:
plt.plot(range(1,len(costs)),costs[1:], "x-")
plt.show()

In [ ]:
# measuring performance
y_pred = (h(X_test,theta0, theta1) >= 0.5)*1
acc = 100-np.sum(np.abs(y_pred-y_test))*100/len(y_test)


print("Die classifcation accuracy ist: ",acc)

# Neurale Netze

Bisher haben wir ein Perzeptron betrachtet, also eine Art Neuronales Netz mit nur einer Schicht. Die gleichen Methoden können wir aber auch für mehrschichtige Neuronale Netze verwenden.

## Forward Pass

**Die Lineare Funktion $f(\cdot)$ unserer Netzwerkschicht erhält folgende Parameter:**
1. Die Matrix $\mathbf{A} \in \mathbb{R}^{n^{[i]}\times{}m}$, mit den Eingabedaten der Netzwerkschicht $i$. Die Zeilen von $\mathbf{A}$ enthalten die einzelnen Datenpunkte. Die Spalten von $\mathbf{A}$ entsprechend den Merkmalen (und damit den Parametern) eines voll-vermaschten Neurons der aktuellen Schicht.
2. Einem Vektor $\mathbf{b} \in \mathbb{R}^{n^{[i]}}$ mit den Bias Parametern der Netzwerkschicht $i$.
3. Die Matrix $\mathbf{W} \in \mathbb{R}^{n^{[i-1]}\times{}n^{[i]}}$ mit den Gewichten der Netzwerkschicht $i$.

**und liefert folgendes Ergebnis $\mathbf{z}$ zurück:** 
$$\mathbf{z} = f_{\mathbf{\theta}}(\mathbf{A})= \mathbf{b} + \mathbf{A}\mathbf{W}$$

Der folgende Screenshot der Folesungsfolien visualisiert die Matrixdimensionen:

<center>
<img src="https://raw.githubusercontent.com/fhswf/aki-ml/refs/heads/main/u6/nn-matrices.png" align="center" style="width: 500px;"/>
<span style="font-size:smaller"><br>
    Abbildung: Visualisierung der Matrix-Dimensionen in verschiedenen Layern $i$.
</span>
</center>


In [ ]:
def f(A, b, W):
    """evaluates linear function.
    Arguments:
        A: Layer input data
        b: Biases vector
        W: Weights matrix
    """
    z = b + A@W
    cache = (A, b, W) # wir merken uns die Werte A, b, und W in jeder Schicht des netzes
    
    return z, cache

**Als Aktivierungsfunktion verwenden wir die (nichtlineare) Sigmoid-Funktion.
Die Funktion erhält die gleichen Parameter wie  `f()` und ruft `f()` auch intern auf:**
$$\mathbf{A}^{[i+1]}= h_{\boldsymbol{\theta}}(\mathbf{A}^{[i]}) = \sigma(\mathbf{z}) = \frac{1}{1+e^{-\mathbf{z}}}$$

In [ ]:
def h(A, b, W):
    """computes the sigmoid of the linear function.
    Arguments:
        A: Layer input data
        b: Biases vector
        W: Weights matrix
    """
    
    z, zcache = f(A, b, W)
    A = 1.0 / (1.0+np.exp(-z))
    cache = (zcache, z) # Zu der Eingabe der Schicht merken wir uns im cache den wert der linearen Funktion f
    
    return A, cache

Wie die Abbildung oben zeigt, ist die Größe des Netzes durch die Anzahl der Neuronen auf jeder Schicht festgelegt.
Kennt man die Anzahl der Neuronen, kann man die Parameter-Matrizen $\mathbf{W}$ und die Vektoren für die Bias Parameter $\mathbf{b}$ initialisieren:

In [ ]:
def init(dims):
    """returns initial weights and biases."""
    #dims = [X,L1,L2,...,y]
    b = []
    W = []
    
    for l in range(1, len(dims)):
        b.append(np.zeros((1, dims[l])))
        W.append(np.random.randn(dims[l-1], dims[l])*0.01) # to break symmetry, it's good to have non-zero weights 
        
    return b, W

Da wir nun die Modellfunktionen für die einezelnen Schichten erstellt haben (und unser Netzwerk auf allen Schichten die gleiche Aktivierungsfunktion nutzt), können wir nun recht einfach eine Funktion für den gesamten **Forward Pass** definieren.
Die Funktion `forward()` erhält als Parameter die Eingabedaten `X`, sowie **Listen** der Bias-Parameter und der Modellparameter für die einzelnen Schichten. Beachten Sie, dass jedes Element der Listen einen Vektor, bzw. eine Matrix mit den Parametern einer Schicht enthält.
Die Länge dieser Listen entspricht also der Tiefe des Neuronalen Netzes.

In [ ]:
def forward(X, b, W):
    """Performs a forward propagation through all layers.
    returns the output layer and cache of intermidiates.
    """
    
    caches = []
    A = X
    
    for l in range(len(b)):
        A_prev = A
        A, cache = h(A_prev, b[l], W[l])
        # Hänge den cache jeder Schicht an die Liste Caches an
        caches.append(cache)
        
    return A, caches

Nachdem wir am Ende des Forward Passes unsere Schätzung berechnet haben, müssen wir für diese Schätzung die Kosten berechnen.
Die Funktion `J()` erhält folgende Parameter:
1. Die vom Modell vorhergesagten Label $\hat{\mathbf{y}}$ in der Größe des Datensatzes `(1, m)`
3. Die tatsächlichen Label unseres Datensatzes $\mathbf{y}$, ebenfalls in der Größe `(1, m)`

**Als Kostenfunktion verwenden wir wie schon bei der Logistischen Regression die Kreuzentropie:**
$$J(\mathbf{W},\mathbf{b})-\frac{1}{m} \sum\limits_{i = 1}^{m} [y^{(i)}\log(\hat{y}^{(i)}) + (1-y^{(i)})\log(1- \hat{y}^{(i)})]$$

In [ ]:
def J(y_pred, y):
    """evaluates Cross-entropy cost function."""
    J = -(y.T@np.log(y_pred) + ((1.0-y.T)@np.log(1.0-y_pred)))/len(y)
    
    return J

## Backpropagation

Der Backpropagation (zu dt. *Fehlerrückführung*) Algorithmus ist ein Spezialfall des Gradientenverfahrens und das am meisten eingesetzte Verfahren zum Trainieren neuronaler Netze.

Um Backpropagation zu implementieren definieren wir eine Funktion mit folgenden Parametern:
1. Den Gradienten der Kosten nach der Aktivierungsfunktion ($\partial A$)
2. Die zwischengespeicherten Werten aus $f(\cdot )$ und $h(\cdot )$ der Aktivierungs- und Übertragungsfunktionen aus dem Forward Pass.

Zurückgeliefert werden die Gradienten $\partial \mathbf{b}$, $\partial \mathbf{W}$, und $\partial \mathbf{A}_{prev}$:

$\partial \mathbf{b}$ ist ein Vektor mit den Gradienten der Bias Parameter: $$\partial \mathbf{b} = \frac{1}{m} \sum_{i=1}^m (\partial {z}^{(i)})$$


$\partial \mathbf{W}$ ist eine Matrix mit den Gradienten der Gewichte: $$ \partial \mathbf{W} = \frac{1}{m} \mathbf{A}_{prev}^T\partial \mathbf{Z}$$

$\partial \mathbf{A}_{prev}$ ist der Grantient der Kostenfunktion bezüglich der Aktivierungsfunktion $\mathbf{A}$ der nächstvorderen Schicht:  $$\partial \mathbf{A}_{prev} = \frac{1}{m}\partial \mathbf{Z} \mathbf{W}^T$$



In [ ]:
def back(dA, cache):
    """calculates grads w.r.t. linear function."""
    
    Zcache, Z = cache

    sigma = 1/(1+np.exp(-Z))
    dZ = dA * sigma * (1-sigma)
        
    A_prev, b, W = Zcache
    
    m = len(A_prev[0])
    db = np.sum(dZ, axis=0, keepdims=True)/m
    dW = (A_prev.T@dZ)/m
    dA_prev = (dZ@W.T)/m
    
    return dA_prev, db, dW

**Wir können nun die Funktion `back()` auf allen Schichten des Modells, beginnend mit der Ausgabeschicht und endend auf der Eingabeschicht aufrufen.**

In [ ]:
def grads(y_hat, y, caches):
    """performs backprobagation through all layers to calculate grads of all parameters."""
    
    db = []
    dW = []
    
    dA = -(y/y_hat - (1-y)/(1-y_hat)) # Gradient der Kostenfunktion auf der Ausgabeschicht
    
    for l in reversed(range(len(caches))):
        # Backward step
        dA, db_, dW_ = back(dA, caches[l])

        db.insert(0, db_)
        dW.insert(0, dW_)
        
    return db, dW

## Optimization

Für das Gradientenverfahren benötigen wir noch eine Funktion, die uns die Parameter jeder Schicht, basierend auf den berechneten Gradienten anpasst.
Diese Funktion `optim` erhält als Parameter:
1. Die Parameter des Neuronalen Netzes ($\mathbf{b}, \mathbf{W}$),
2. die Gradienten der einzelnen Schichten ($\partial \mathbf{b}, \partial \mathbf{W}$) 
3. sowie die Lernrate $\alpha$.

Die Funktion liefert die aktualisierten Parameter ($\mathbf{b}, \mathbf{W}$) zurück.

In [ ]:
def optim(parameters, grads, alpha):
    """updates parameters using gradient decent updating rule."""
    
    b, W = parameters
    db, dW = grads
    
    for l in range(len(b)):
        b[l] = b[l] - alpha*db[l]
        W[l] = W[l] - alpha*dW[l]
        
    return b, W

## Ein Modell Trainieren

Wir können nun unser selbst programmiertes MLP Netzwerk mit einem echten Datensatz trainieren. Wir verwenden hier den bekannten MNIST Datensatz.

In [ ]:
try:
    from tensorflow.keras import datasets , utils
except:
    import sys
    !{sys.executable} -m pip install --upgrade pip
    !{sys.executable} -m pip install tensorflow
    #!{sys.executable} -m pip install tensorflow-macos tensorflow-metal
    from tensorflow.keras import datasets , utils

In [ ]:
#Importing MNIST from tensorflow dataset liabrary

(X_train, y_train), (X_test, y_test) = datasets.mnist.load_data()

Wenn wir den Datensatz geladen haben, können wir uns einige Beispiele der Daten anzeigen lassen:

In [ ]:
fig, ax = plt.subplots(1, 10, figsize=(18,3))

for i in range(10):
    ax[i].set_title('Image label: %d' %i)
    ax[i].axis('off')
    ax[i].imshow(X_train[y_train == i][0], cmap='gray')
plt.show()

Für unser MLP Netzwerk machen wir aus den $28\times{}28$ Pixel großen Bildern der Ziffern eindimensionale Vektoren der Länge $748$.

In [ ]:
# preprocessing
print("Vorher:", X_train.shape)
X_train = X_train.reshape(-1, 28*28)/255
X_test = X_test.reshape(-1, 28*28)/255
y_train = utils.to_categorical(y_train)
y_test = utils.to_categorical(y_test)
print("Nachher:", X_train.shape)

Nun stellen wir aus allen zuvor definierten Funktionen unser Modell auf:

In [ ]:
def model(X, y, layers_dims, alpha = 0.01, epoch = 20, batch_size = 400):
    """MLP, das die Softmax-Kostenfunktion mit dem Gradientenverfahren optimiert.
    X: Eingabedatem
    y: Labels
    layers_dims: Liste mit en Dimensionen der Schichten
    alpha(default = 0.01): Lernrate
    epoch(default = 20): Anzahl der zu trainierenden Epochen
    batch_size(default = 400): Größe eines Minibatches
    """
    
    costs = []
    parameters = init(layers_dims)
    b, W = parameters
    
    for i in range(epoch):
        
        for batch in range(0,len(y),batch_size):
            y_hat, caches = forward(X[batch:batch+batch_size], b, W)
            cost = J(y_hat, y[batch:batch+batch_size])
            gradss = grads(y_hat, y[batch:batch+batch_size], caches)
            parameters = optim(parameters, gradss, alpha)
            b, W = parameters

        costs.append(cost.item(0))
        print(f'Epoch {i}: {cost.item(0)}')
            
    plt.plot(range(1,len(costs)),costs[1:], "x-")
    plt.show()
    return parameters

In [ ]:
trained_weights_10 = model(X_train, y_train, [28*28, 10], 0.5)

In [ ]:
class mlp_model:
    
    def __init__(self, X, y, layers_dims):
        """MLP Modell
        X: Eingabedatem
        y: Labels
        layers_dims: Liste mit en Dimensionen der Schichten
        """
        self.X = X
        self.y = y
        self.costs = []
        self.parameters = init(layers_dims)
        
    def fit(self, alpha = 0.01, epoch = 20, batch_size = 400):
        """MLP, das die Softmax-Kostenfunktion mit dem Gradientenverfahren optimiert.
        alpha(default = 0.01): Lernrate
        epoch(default = 20): Anzahl der zu trainierenden Epochen
        batch_size(default = 400): Größe eines Minibatches
        """
        
        b, W = self.parameters
    
        for i in range(epoch):
        
            for batch in range(0,len(self.y),batch_size):
                y_hat, caches = forward(self.X[batch:batch+batch_size], b, W)
                cost = J(y_hat, self.y[batch:batch+batch_size])
                gradss = grads(y_hat, self.y[batch:batch+batch_size], caches)
                self.parameters = optim(self.parameters, gradss, alpha)
                b, W = self.parameters

            self.costs.append(cost.item(0))
            print(f'Epoch {i}: {cost.item(0):.5}')
            
        plt.plot(range(1,len(self.costs)),self.costs[1:], "x-")
        plt.show()
        return self.parameters
    
    def predict(self, X):
        """Vorhersagefunktion für das trainierte Modell."""
    
        b, W = self.parameters
        y_pred, _ = forward(X, b, W)
    
        return (y_pred>=0.5)*1
    
    def accuracy(y_test, y_pred_0):
        return 100-np.sum(np.abs(y_pred_0-y_test))*100/len(y_test)

In [ ]:
model_10 = mlp_model(X_train, y_train, [28*28, 10])
trained_weights_10 = model_10.fit(0.5)

In [ ]:
y_pred_10 = model_10.predict(X_test)
mlp_model.accuracy(y_test, y_pred_10)

In [ ]:
model_20_10 = mlp_model(X_train, y_train, [28*28, 20, 10])
trained_weights_20_10 = model_20_10.fit(0.5)

In [ ]:
y_pred_20_10 = model_20_10.predict(X_test)
mlp_model.accuracy(y_test, y_pred_20_10)

# Neuronale Netze mit Keras

In dieser Aufgabenblatt geht es darum, ein einfachen Künstliches Neuronales Netz für ein Regressionsproblem mit der Keras API zu implementieren.

Wir verwenden den *Boston Housing* Datensatz, der als Standard-Anwendungsbeispiel über die Keras API heruntergeladen werden kann.

Der Datensatz beschreibt die Wohnverhältnisse in verschieden Gebieten um Boston in den 1970er Jahren.
Er enthält 506 Einträge mit jeweils 13 numerischen Merkmalen, die Zielvariable beschreibt den mittleren Wert der Häuserpreise in dem jeweiligen Bezirk (in Tausend USD).

Gute Modelle sollten Vorhersagen mit einer mittlere quadratische Abweichung (*mean squared error*, MSE) unterhalb 20 (Tausend USD) treffen.



Zuerst binden wir die wichtigsten Bibliotheken ein und laden den Datensatz mit der Funktion `boston_housing.load_data()` herunter.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import models, layers, datasets

In [ ]:
(X_train, y_train), (X_test, y_test) = datasets.boston_housing.load_data()

In [ ]:
mean = X_train.mean(axis=0)
X_train -= mean
std = X_train.std(axis=0)
X_train /= std

X_test -= mean
X_test /= std

X_test.shape

In [ ]:
model = models.Sequential()
model.add(layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(1))

Nachdem die Modellparameter zusammengestellt sind, muss die Eigentliche Modellfunktion erstellt werden.
In Tensorflow wird damit der Graph des Modells erzeugt.
Die `compile`-Funktion kann mit verschiedenen Parametern angepasst werden:
- `optimizer` legt die Art (den Algorithmus) des Gradientenverfahrens fest. (https://keras.io/optimizers/)
- `loss` bestimmt die Art der Kostenfunktion (https://keras.io/losses/)
- `metrics` legt fest, welche Qualitätsmerkmale beim Trainieren überprüft werden sollen (https://keras.io/metrics/)



In [ ]:
model.compile(
    optimizer='SGD',
    loss='mse',
    metrics=['mae','mape'])

Um die Ergebnisse zu visulaisieren, legen wir ein Log-Verzeichnis pro Lauf des Verfahrens an, in das wir die Informationen der `fit`-Funktion schreiben.
Damit können wir später die Ergebnisse der einzelnen Trainingsläufe vergeleichen.

In [ ]:
import time
newlog = '.\\keras_data\\run_' + time.strftime("%Y%m%d-%H%M%S")
tbCallBack = tf.keras.callbacks.TensorBoard(log_dir=newlog, histogram_freq=0, write_graph=True, write_images=True)

Nun rufen wir die `fit`-Funktion auf, um die Modellparameter zu lernen.

In [ ]:
model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=64,
    callbacks=[tbCallBack]
)

Nach dem Training verwenden wir die Testdaten, um unser trainiertes Modell zu evaluieren

In [ ]:
mse_score, mae_score, mape_score = model.evaluate(X_test, y_test, verbose=False)
print("Mittlwert der Fehlerquadrate: ", mse_score)
print("Mittlerer absoluter Fehler: ", mae_score)
print("Mittlerer absoluter Fehler (in Prozent): ", mape_score)

Das Trainierte Modell schätzt die Häuserpreise vermutlich nicht sonderlich gut.
Dies liegt an der sehr unglücklichen Wahl der (Hyper-)Parameter.
Versuchen Sie die Parameter sinnvoll anzupassen, sodass sich die Qualität der Schaätzfunktion verbessert.

Um die Log-Dateien sowie den Modellgraphnen zu visualisieren, können die Tensorboard verwenden.
Rufen Sie daszu (im Verzeichnis der *.ipynb* Datei) das Kommando `tensorboard --logdir=./keras_data` auf.
Wenn Tensorboard gestartet ist, erreichen Sie die Web-Oberfläche unter der URL `localhost:8008`.

In [ ]:
for a in model.get_weights():
    print(np.shape(a))

In [ ]:
model.summary()

Wir verwenden nun in einem zweiten Beispiel Keras mit dem MNIST Datensatz:

### Beschaffen der Daten und Vorverarbeiten

1. Daten mit `datasets.mnist.load_data()` laden
2. Bilder in eindimensionale Arrays umwandeln mit `reshape`. Teilen durch 255 um zu Normalisieren.
3. One-Hot Encoding der Labels mit `utils.to_categorical`

In [ ]:
# Data loading
(X_train, y_train), (X_test, y_test) = (None, None), (None, None)
# YOUR CODE HERE
raise NotImplementedError()


# Data preprocessing
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# Test Cell
#----------

assert X_train.ptp() == X_test.ptp() == 1, 'Data is not nomalized'
assert X_train.shape == (60000, 784)
assert X_test.shape == (10000, 784)
assert y_train.shape == (60000, 10), 'Make sure to one hot encode train labels'
assert y_test.shape == (10000, 10), 'Make sure to one hot encode test labels'

### Das Modell aufstellen
<div class="alert alert-block alert-info">
<b>Aufgabe:</b> Erstellen Sie ein Modell mit geeigneten Hyperparametern.

*Hinweise:*
- Verwenden Sie die `softmax` Aktivierungsfunktion um die Ausgabeschicht in eine Wahrscheinlichkeitsfunktion umzuwandeln.
- `input_shape` ist ein Vektor der Dimension `(784,)`
</div>

In [ ]:
mnist_model = None

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# Test Cell
#----------

assert type(mnist_model) == models.Sequential
assert mnist_model.built, 'The model is not built'
assert len(mnist_model.layers) > 2, 'You should have at least one hidden layer'
assert mnist_model.input_shape[1:] == X_train[0].shape, 'Input must match the features of the image'
assert mnist_model.output_shape[1] == 10, 'Output must match the number of classes'

### Modell Kompilieren
Verwenden Sie `compile` um den `optimizer`, die `loss` Funktion und die `metrics` festzulegen.

*Hinweise:* 
- Da wir `softmax` verwenden, wählen wir die Kostenfunktion `categorical_crossentropy`.
- `accuracy` ist an dieser Stelle eine gute Metrik für die Klassifikation

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# Test Cell
#----------


assert mnist_model.loss
assert mnist_model.optimizer

### Training
Trainieren Sie das Modell mit der `fit` Funktion.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# Test Cell
#----------

assert len(mnist_model.history.epoch) >= 2

### Testen

<div class="alert alert-block alert-info">
Evaluieren Sie das Modell mit der <code>evaluate()</code> Funktion.
</div>

In [ ]:
loss, acc = [None]*2

# YOUR CODE HERE
raise NotImplementedError()
print("loss: %.4f - accuracy: %.4f " %(loss, acc))

### Visualisierung

In [ ]:
img_i = np.random.randint(10000, size=8)
fig, ax = plt.subplots(1, 8, figsize=(18,3))
for i, j in enumerate(img_i):
    ax[i].set_title('True: %d\nPredicted: %d' %(np.argmax(y_test[j]),
                np.argmax(mnist_model.predict(X_test[j].reshape(1, -1)), axis=-1)))
    ax[i].axis('off')
    ax[i].imshow(X_test[j].reshape(28,28), cmap='gray')
plt.show()

In [ ]:
mnist_model.summary()

## Referenzen:
[1] M. Berthold, C. Borgelt, F. Höppner and F. Klawonn, Guide to Intelligent Data Analysis, London: Springer-Verlag, 2010.  
[2] A. Ng, Machine Learning Yearning, deeplearning.ai, 2018.  

**Tipp:** Schauen Sie sich die Playlist von *3Blue1Brown* über Neuronale Netze auf [Youtube](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi) an

Beiträge zu diesem Notebook: [Heiner Giefers](https://github.com/hgiefers), [Stefan Goetze](https://github.com/Stefan-Goe)